### Demo: Data preparation for BERT model fine-tuning

#### Step 1: Imports

In [ ]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

#### Step 2: Configurations

In [14]:
MAX_LENGTH = 128
CORONA_DATASET_PATH = "data/Corona_NLP_train.csv" 
TEST_SPLIT_FRACTION = 0.1
RANDOM_SEED = 100
BATCH_SIZE = 32
TOKENIZATION_BATCH_SIZE = 256
DATASET_COLUMNS = ["OriginalTweet", "Sentiment"]

#### Step 3: Prepare the dataset
- Tokenize the dataset
- Partition the dataset

In [15]:
class DataPreprocessor:
    def __init__(self, dataset_path=CORONA_DATASET_PATH, max_length=MAX_LENGTH):
        self.dataset_path = dataset_path
        self.max_length = max_length
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.label_encoder = LabelEncoder()
        self.dataset = self.load_data()
        self.tokenized_data = self.tokenize_text()
        self.input_ids = None
        self.attention_masks = None
        self.labels = None

    def load_data(self):        
        data = pd.read_csv(self.dataset_path, encoding='latin-1')[DATASET_COLUMNS].values.tolist()
        return data
    
    def tokenize_text(self, batch_size=TOKENIZATION_BATCH_SIZE):
        texts = [str(text) for text, _ in self.dataset]
        tokenized_batches = []

        for start in tqdm(
            range(0, len(texts), batch_size),
            desc='Tokenizing tweets',
            unit='batch'
        ):
            batch = self.tokenizer(
                texts[start:start + batch_size],
                padding='max_length',
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )
            tokenized_batches.append(batch)

        return {
            key: torch.cat([batch[key] for batch in tokenized_batches], dim=0)
            for key in tokenized_batches[0]
        }

    def preprocess_data(self, data=None):
        self.input_ids = self.tokenized_data['input_ids']
        self.attention_masks = self.tokenized_data['attention_mask']
        self.labels = torch.tensor(self.label_encoder.fit_transform(
            [label for _, label in self.dataset]))
        train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = self.split_data()
        return train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask

    def split_data(
        self, 
        test_size=TEST_SPLIT_FRACTION, random_state=RANDOM_SEED):
        train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = train_test_split(
            self.input_ids, self.labels, self.attention_masks,
            random_state=random_state, test_size=test_size
        )
        return train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask

#### Step 4: Create data loaders
- Create train/validate data loaders to present the dataset to the BERT model

In [4]:
class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask': 
                 self.attention_mask[idx],
                'labels': self.labels[idx]}
        
def create_dataloaders(
    train_inputs, 
    val_inputs, 
    train_labels, 
    val_labels, 
    train_mask, 
    val_mask, 
    batch_size=BATCH_SIZE):
    train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

#### Step 5: Create BERT classification model

In [20]:
data_preprocessor = DataPreprocessor()
train_val_splits = data_preprocessor.preprocess_data(data_preprocessor.dataset)
train_loader, val_loader = create_dataloaders(*train_val_splits)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(set(data_preprocessor.labels)))

Tokenizing tweets: 100%|██████████| 161/161 [00:30<00:00,  5.31batch/s]
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
c:\Users\adeid\miniconda3\envs\slm_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adeid\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as 